<a href="https://colab.research.google.com/github/Krishishah7/nlp-learning-series/blob/main/06_llm_and_fine_tuning/04_self_consistency_prompting/self_consistency_prompting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U transformers accelerate sentencepiece --quiet

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-large"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [ ]:
question = "John is taller than Mary. Mary is taller than Sam. Who is the tallest?"

In [20]:
prompt_cot = f"""
Solve step by step:

{question}
"""

inputs = tokenizer(prompt_cot, return_tensors="pt")

outputs = model.generate(
    **inputs,
    max_new_tokens=60,
    temperature=0.7,
    do_sample=True
)

print("SINGLE CoT OUTPUT:")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

SINGLE CoT OUTPUT:
Mary is taller than John. John is taller than Mary. The answer: John.


In [21]:
num_samples = 5
answers = []

for _ in range(num_samples):
    outputs = model.generate(
        **inputs,
        max_new_tokens=60,
        temperature=0.9,
        do_sample=True
    )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answers.append(text)

print("MULTIPLE REASONING PATHS:\n")
for i, ans in enumerate(answers, 1):
    print(f"Sample {i}:\n{ans}\n")

MULTIPLE REASONING PATHS:

Sample 1:
Mary is taller than Sam and John is taller than Mary so they are both tall. Sam is shorter than John and Mary so John is taller than Mary. The answer: John.

Sample 2:
Mary and John are taller than each other. So Mary is taller than Sam. The answer is Sam.

Sample 3:
Mary is taller than Mary, so she is taller. John is taller than Mary, so he is taller. The answer: John.

Sample 4:
Mary is taller than Sam, so John is taller than Sam. The answer: John.

Sample 5:
Mary is taller than Sam. John is taller than Mary. If John is taller than Mary, then he is the taller person. The answer: John.



In [22]:
from collections import Counter

# Extract final answers (last sentence of each output)
final_answers = []

for ans in answers:
    # Take last sentence as final answer
    parts = ans.strip().split(".")
    last_sentence = parts[-2] if len(parts) > 1 else ans
    final_answers.append(last_sentence.strip())

# Count occurrences
most_common = Counter(final_answers).most_common(1)

print("FINAL ANSWER BY SELF-CONSISTENCY:")

if most_common:
    print(most_common[0][0])
else:
    print("No valid answer extracted")

FINAL ANSWER BY SELF-CONSISTENCY:
The answer: John
